In [0]:
import os
from pyspark.sql.functions import col, hour, month

os.environ["SPARKML_TEMP_DFS_PATH"] = "/Volumes/workspace/default/mama/sparkml_cache"

raw_df = spark.table("workspace.default.us_accidents_hpms_final_100_m_features")

print(raw_df.count(), len(raw_df.columns))
display(raw_df.limit(5))

4484584 50


Severity,Start_Time,Start_Lat,Start_Lng,Distance(mi),City,County,State,Temperature(F),Humidity(%),Pressure(in),Visibility(mi),Wind_Speed(mph),Precipitation(in),Weather_Condition,Sunrise_Sunset,AADT,AADT_COMBINATION,AADT_SINGLE_UNIT,ACCESS_CONTROL,FACILITY_TYPE,F_SYSTEM,SPEED_LIMIT,SURFACE_TYPE,StateID,THROUGH_LANES,URBAN_ID,hpms_distance_m,Hour,Month,DayOfWeek,Is_Weekend,Is_Peak_Hour,Is_Night,Is_Rain,Is_Snow,Is_Fog,Low_Visibility,High_Wind,Has_Precipitation,Truck_AADT,Truck_Ratio,AADT_per_Lane,Is_High_Speed,Is_Controlled_Access,Is_Urban,Very_Close_HPMS,Rain_LowVis,Snow_LowVis,HighSpeed_Urban
2,2022-05-26T18:24:00.000Z,38.794748,-77.14471800000003,5.749,Alexandria,Fairfax County,VA,72.0,83.0,30.03,10.0,6.0,0.0,Cloudy,Day,172980.0,13470.0,3407.0,1.0,2,1.0,55.0,7.0,51,11.0,92242,1.500479455243109,18.0,5.0,3.0,0,1,0,0,0,0,0,0,0,16877.0,0.097566195,15725.454,0,1,1,1,0,0,0
2,2022-03-01T16:00:00.000Z,38.795042,-77.18656999999997,4.807,Springfield,Fairfax County,VA,56.0,35.0,29.91,10.0,8.0,0.0,Fair,Day,null,null,null,null,6,1.0,null,7.0,51,null,92242,0.8331936120027094,16.0,3.0,1.0,0,1,0,0,0,0,0,0,0,0.0,null,null,0,0,1,1,0,0,0
2,2022-08-28T13:00:30.000Z,37.325115,-80.033187,4.323,Roanoke,Roanoke County,VA,86.0,61.0,28.9,10.0,9.0,0.0,Partly Cloudy,Day,null,null,null,null,6,1.0,null,2.0,51,null,75421,1.660423180908143,13.0,8.0,6.0,1,0,0,0,0,0,0,0,0,0.0,null,null,0,0,1,1,0,0,0
2,2022-09-15T17:27:59.000Z,38.939907,-77.384456,1.749,Herndon,Fairfax County,VA,78.0,42.0,29.81,10.0,9.0,0.0,Fair,Day,null,null,null,null,6,3.0,null,null,51,null,92242,2.1592777364387974,17.0,9.0,3.0,0,1,0,0,0,0,0,0,0,0.0,null,null,0,0,1,1,0,0,0
2,2022-08-19T16:19:29.000Z,38.745025,-77.250609,0.029,Springfield,Fairfax,VA,86.0,52.0,29.96,10.0,9.0,0.0,Partly Cloudy,Day,null,null,null,null,6,3.0,null,null,51,null,92242,53.23505290330992,16.0,8.0,4.0,0,1,0,0,0,0,0,0,0,0.0,null,null,0,0,1,0,0,0,0


In [0]:
use_df = raw_df.limit(500000).select(
    col("Severity").cast("int").alias("Severity"),
    col("Distance(mi)").cast("double").alias("Distance_mi"),
    col("Temperature(F)").cast("double").alias("Temperature_F"),
    col("Humidity(%)").cast("double").alias("Humidity_pct"),
    col("Visibility(mi)").cast("double").alias("Visibility_mi"),
    col("Wind_Speed(mph)").cast("double").alias("Wind_Speed_mph"),
    col("Precipitation(in)").cast("double").alias("Precipitation_in"),
    hour(col("Start_Time")).alias("Hour"),
    month(col("Start_Time")).alias("Month"),
    col("AADT").cast("double").alias("AADT"),
    col("SPEED_LIMIT").cast("double").alias("SPEED_LIMIT"),
    col("THROUGH_LANES").cast("double").alias("THROUGH_LANES"),
    col("URBAN_ID").cast("double").alias("URBAN_ID"),
    col("hpms_distance_m").cast("double").alias("hpms_distance_m")
).dropna()

use_df = use_df.withColumn("label", (col("Severity") - 1).cast("double"))

print("rows ready:", use_df.count())
display(use_df.limit(5))

rows ready: 386010


Severity,Distance_mi,Temperature_F,Humidity_pct,Visibility_mi,Wind_Speed_mph,Precipitation_in,Hour,Month,AADT,SPEED_LIMIT,THROUGH_LANES,URBAN_ID,hpms_distance_m,label
2,5.749,72.0,83.0,10.0,6.0,0.0,18,5,172980.0,55.0,11.0,92242.0,1.500479455243109,1.0
2,1.113,37.0,89.0,3.0,12.0,0.2,12,12,70020.0,60.0,4.0,75421.0,0.4524617631343032,1.0
2,0.8759999999999999,79.0,56.0,10.0,6.0,0.0,12,5,70020.0,60.0,4.0,75421.0,1.1725322867301835,1.0
2,1.245,74.0,70.0,10.0,10.0,0.0,16,9,56098.0,70.0,4.0,37162.0,0.5794107499909252,1.0
2,2.335,23.0,88.0,1.0,14.0,0.0,5,11,9559.0,60.0,4.0,99999.0,0.2526683416320272,1.0


In [0]:
use_df.groupBy("label").count().orderBy("label").show()

+-----+------+
|label| count|
+-----+------+
|  0.0|  5745|
|  1.0|273633|
|  2.0| 98651|
|  3.0|  7981|
+-----+------+



In [0]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

cols = [
    "Distance_mi",
    "Temperature_F",
    "Humidity_pct",
    "Visibility_mi",
    "Wind_Speed_mph",
    "Precipitation_in",
    "Hour",
    "Month",
    "AADT",
    "SPEED_LIMIT",
    "THROUGH_LANES",
    "URBAN_ID",
    "hpms_distance_m"
]

assembler = VectorAssembler(inputCols=cols, outputCol="features")
scaler = StandardScaler(inputCol="features", outputCol="scaled")

lr = LogisticRegression(
    featuresCol="scaled",
    labelCol="label",
    family="multinomial",
    maxIter=20
)

pipe = Pipeline(stages=[assembler, scaler, lr])

param_grid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0.01, 0.1, 1.0]) \
    .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0]) \
    .build()

evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

cv = CrossValidator(
    estimator=pipe,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    numFolds=3,
    seed=42,
    parallelism=1
)

In [0]:
train_df, test_df = use_df.randomSplit([0.8, 0.2], seed=42)

print("train:", train_df.count())
print("test:", test_df.count())

train_df.groupBy("label").count().orderBy("label").show()
test_df.groupBy("label").count().orderBy("label").show()

train: 308874
test: 77136
+-----+------+
|label| count|
+-----+------+
|  0.0|  4597|
|  1.0|219126|
|  2.0| 78779|
|  3.0|  6372|
+-----+------+

+-----+-----+
|label|count|
+-----+-----+
|  0.0| 1148|
|  1.0|54507|
|  2.0|19872|
|  3.0| 1609|
+-----+-----+



In [0]:
import mlflow

with mlflow.start_run(run_name="Phase3_LogisticRegression_HPMS_4class_500k"):
    print("training...")

    cv_model = cv.fit(train_df)
    best_lr = cv_model.bestModel.stages[-1]

    print("best regParam:", best_lr.getRegParam())
    print("best elasticNetParam:", best_lr.getElasticNetParam())

    mlflow.log_param("regParam", best_lr.getRegParam())
    mlflow.log_param("elasticNetParam", best_lr.getElasticNetParam())
    mlflow.log_param("model_type", "LogisticRegression_4class")

training...
best regParam: 0.01
best elasticNetParam: 0.0


In [0]:
import pandas as pd
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

best_model = cv_model.bestModel
pred = best_model.transform(test_df)

acc_eval = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

f1_eval = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

wp_eval = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedPrecision"
)

wr_eval = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedRecall"
)

results_df = pd.DataFrame({
    "Metric": ["Accuracy", "F1 Score", "Weighted Precision", "Weighted Recall"],
    "Value": [
        round(acc_eval.evaluate(pred), 4),
        round(f1_eval.evaluate(pred), 4),
        round(wp_eval.evaluate(pred), 4),
        round(wr_eval.evaluate(pred), 4)
    ]
})

print(results_df.to_string(index=False))

            Metric  Value
          Accuracy 0.7413
          F1 Score 0.7032
Weighted Precision 0.7044
   Weighted Recall 0.7413


In [0]:
pred.groupBy("label", "prediction").count().orderBy("label", "prediction").show()
display(pred.select("label", "prediction", "probability").limit(20))

+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       1.0| 1062|
|  0.0|       2.0|   86|
|  1.0|       1.0|50476|
|  1.0|       2.0| 4016|
|  1.0|       3.0|   15|
|  2.0|       1.0|13177|
|  2.0|       2.0| 6693|
|  2.0|       3.0|    2|
|  3.0|       1.0| 1536|
|  3.0|       2.0|   64|
|  3.0|       3.0|    9|
+-----+----------+-----+



label,prediction,probability
0.0,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.01053631737347543"",""0.9223513793830073"",""0.05856719459121678"",""0.008545108652300475""]}"
0.0,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.021703345559485478"",""0.8861454610903567"",""0.07128208657247295"",""0.02086910677768477""]}"
0.0,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.02104240502807387"",""0.8973804800666767"",""0.057399484731063016"",""0.024177630174186244""]}"
0.0,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.01317309341603774"",""0.7606266168235667"",""0.196916991679031"",""0.02928329808136461""]}"
0.0,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.00946236079930467"",""0.6015896340019186"",""0.3768293709816987"",""0.012118634217077935""]}"
0.0,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.012848500328602151"",""0.7950282682053482"",""0.14555219307521147"",""0.04657103839083826""]}"
0.0,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.012717783124642197"",""0.7655903682554526"",""0.17455896955110106"",""0.04713287906880436""]}"
0.0,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.015649962083185805"",""0.8660336119681643"",""0.09517309987690419"",""0.02314332607174557""]}"
0.0,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.02313882748177486"",""0.8850782887820181"",""0.06802760169107462"",""0.023755282045132248""]}"
0.0,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.004165552559708243"",""0.6297815972332381"",""0.3590430005356372"",""0.0070098496714164574""]}"


In [0]:
model_path = "/Volumes/workspace/default/mama/models/logistic_regression_hpms_4class_500k"
best_model.write().overwrite().save(model_path)

print("saved to:", model_path)

+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       1.0| 1062|
|  0.0|       2.0|   86|
|  1.0|       1.0|50476|
|  1.0|       2.0| 4016|
|  1.0|       3.0|   15|
|  2.0|       1.0|13177|
|  2.0|       2.0| 6693|
|  2.0|       3.0|    2|
|  3.0|       1.0| 1536|
|  3.0|       2.0|   64|
|  3.0|       3.0|    9|
+-----+----------+-----+



label,prediction,probability
0.0,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.01053631737347543"",""0.9223513793830073"",""0.05856719459121678"",""0.008545108652300475""]}"
0.0,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.021703345559485478"",""0.8861454610903567"",""0.07128208657247295"",""0.02086910677768477""]}"
0.0,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.02104240502807387"",""0.8973804800666767"",""0.057399484731063016"",""0.024177630174186244""]}"
0.0,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.01317309341603774"",""0.7606266168235667"",""0.196916991679031"",""0.02928329808136461""]}"
0.0,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.00946236079930467"",""0.6015896340019186"",""0.3768293709816987"",""0.012118634217077935""]}"
0.0,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.012848500328602151"",""0.7950282682053482"",""0.14555219307521147"",""0.04657103839083826""]}"
0.0,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.012717783124642197"",""0.7655903682554526"",""0.17455896955110106"",""0.04713287906880436""]}"
0.0,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.015649962083185805"",""0.8660336119681643"",""0.09517309987690419"",""0.02314332607174557""]}"
0.0,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.02313882748177486"",""0.8850782887820181"",""0.06802760169107462"",""0.023755282045132248""]}"
0.0,1.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.004165552559708243"",""0.6297815972332381"",""0.3590430005356372"",""0.0070098496714164574""]}"
